# Example 0.13.1 / 2: Exercise 3-1: Dantzig's Transportation Problem

Companion notebook for **0.13.1 Exercises on Building Models** in the AMPL lab Documentation, section "Exercise 3-1: Dantzig's Transportation Problem". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 3-1: Dantzig's Transportation Problem

Plants in Seattle and San Diego can supply 350 and 600 cases per week. Customers in New York,
Chicago and Topeka order 325, 300 and 275 cases. Shipping costs \$90 per case per thousand miles,
over these distances in miles:

| | New York | Chicago | Topeka |
|---|---|---|---|
| Seattle | 2500 | 1700 | 1800 |
| San Diego | 2500 | 1800 | 1400 |

Supply exceeds demand (950 against 900), so supply is an upper limit and demand must be met. The
cost per case is a computed parameter, so the distances stay as given
(Computed Parameters (AMPL lab documentation, section 1.2.2)); the rest is the model of
The Transportation Model (AMPL lab documentation, section 0.9.2).

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIG;                                   # plants
set DEST;                                   # customers
param supply {ORIG} >= 0;                   # cases per week available
param demand {DEST} >= 0;                   # cases per week ordered
param miles {ORIG, DEST} >= 0;              # distance in miles
param rate >= 0;                            # dollars per case per thousand miles
param cost {i in ORIG, j in DEST} = rate * miles[i,j] / 1000;  # dollars per case

var Ship {ORIG, DEST} >= 0;                 # cases shipped per week

minimize Total_Cost: sum {i in ORIG, j in DEST} cost[i,j] * Ship[i,j];  # dollars per week

subject to Supply {i in ORIG}: sum {j in DEST} Ship[i,j] <= supply[i];  # within plant capacity
subject to Demand {j in DEST}: sum {i in ORIG} Ship[i,j] = demand[j];  # each order met

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ORIG"] = ["Seattle", "SanDiego"]
ampl.set["DEST"] = ["NewYork", "Chicago", "Topeka"]
ampl.param["supply"] = {"Seattle": 350, "SanDiego": 600}
ampl.param["demand"] = {"NewYork": 325, "Chicago": 300, "Topeka": 275}
ampl.param["miles"] = {
    ("Seattle", "NewYork"): 2500, ("Seattle", "Chicago"): 1700, ("Seattle", "Topeka"): 1800,
    ("SanDiego", "NewYork"): 2500, ("SanDiego", "Chicago"): 1800, ("SanDiego", "Topeka"): 1400,
}
ampl.param["rate"] = 90

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print({k: v for k, v in ampl.var["Ship"].to_dict().items() if v > 0})
print("cost per week:", ampl.obj["Total_Cost"].value())

HiGHS 1.14.0: optimal solution; objective 153675
1 simplex iterations
0 barrier iterations
solved
{('SanDiego', 'NewYork'): 275, ('SanDiego', 'Topeka'): 275, ('Seattle', 'Chicago'): 300, ('Seattle', 'NewYork'): 50}
cost per week: 153675.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ORIG := Seattle SanDiego;
set DEST := NewYork Chicago Topeka;
param supply := Seattle 350  SanDiego 600;
param demand := NewYork 325  Chicago 300  Topeka 275;
param miles:  NewYork  Chicago  Topeka :=
Seattle          2500     1700    1800
SanDiego         2500     1800    1400;
param rate := 90;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Ship, Total_Cost;

HiGHS 1.14.0: optimal solution; objective 153675
1 simplex iterations
0 barrier iterations
Ship :=
SanDiego Chicago     0
SanDiego NewYork   275
SanDiego Topeka    275
Seattle  Chicago   300
Seattle  NewYork    50
Seattle  Topeka      0
;

Total_Cost = 153675

